# TokenVector.Inference vs PyTorch (Colab T4 GPU)

GPT-2 124M, seq1024. Chạy theo thứ tự từ trên xuống.

Loss `.tkv` ≠ torch (FLOP giống, init khác) — **chỉ so time**.


In [ ]:
!nvidia-smi


In [ ]:
# 1/5 Install .NET 10 SDK
import urllib.request
!apt-get update -qq && apt-get install -y -qq build-essential gcc libicu74 libssl3 libkrb5-3 zlib1g >/dev/null 2>&1 || apt-get install -y -qq build-essential gcc libicu-dev libkrb5-dev >/dev/null
urls = [
    'https://dot.net/v1/dotnet-install.sh',
    'https://builds.dotnet.microsoft.com/dotnet/scripts/v1/dotnet-install.sh',
]
for u in urls:
    try:
        with urllib.request.urlopen(u, timeout=30) as r:
            open('/tmp/dotnet-install.sh', 'wb').write(r.read())
        print('Downloaded .NET install script from:', u)
        break
    except Exception as e:
        print('FAIL', u, e)
else:
    raise RuntimeError('Không tải được dotnet-install.sh')
!bash /tmp/dotnet-install.sh --channel 10.0 --install-dir /opt/dotnet
import os
os.environ['PATH'] = '/opt/dotnet:' + os.environ['PATH']


In [ ]:
# 2/5 Clone repos + cài LD_LIBRARY_PATH cho torch CUDA
import os, glob, subprocess
for repo in ['TokenVector', 'TokenVector.Inference']:
    p = subprocess.run('git clone --depth 1 https://github.com/nguyenhungtran18/{}.git 2>&1 || (cd {} && git pull --depth 1 2>&1)'.format(repo, repo), shell=True, capture_output=True, text=True)
    print(repo, '->', 'ok' if p.returncode == 0 else 'fail: ' + p.stderr[-300:])
cublas_dir = next(iter(glob.glob('/usr/local/lib/python*/dist-packages/nvidia/cublas/lib')), None)
cudart_dir = next(iter(glob.glob('/usr/local/lib/python*/dist-packages/nvidia/cuda_runtime/lib')), None)
torch_dir  = next(iter(glob.glob('/usr/local/lib/python*/dist-packages/torch')), None)
print('cublas_dir:', cublas_dir)
print('cudart_dir:', cudart_dir)
print('torch     :', torch_dir)
ld = []
if cublas_dir: ld.append(cublas_dir)
if cudart_dir: ld.append(cudart_dir)
if torch_dir:  ld.append(torch_dir + '/lib')
os.environ['LD_LIBRARY_PATH'] = ':'.join(ld) + (':' + os.environ.get('LD_LIBRARY_PATH', '') if os.environ.get('LD_LIBRARY_PATH') else '')
print('LD_LIBRARY_PATH =', os.environ['LD_LIBRARY_PATH'])


In [ ]:
# 3/5 Build tkv_simd_native.so + tkv_step124
import subprocess
r = subprocess.run('cd /content/TokenVector/Release/tools/simd_native && gcc -O2 -mavx2 -mfma -shared -fPIC tkv_simd_native.c tkv_cuda_native.c tkv_linear_dw.c tkv_bwd_ops.c tkv_flash_attn.c tkv_step124.c tkv_prof.c tkv_attn_cuda.c -lm -lpthread -ldl -o tkv_simd_native.dll', shell=True, capture_output=True, text=True)
print('native build rc =', r.returncode)
if r.returncode != 0: print(r.stderr[-3000:])
r = subprocess.run('cp /content/TokenVector/Release/tools/simd_native/tkv_simd_native.dll /content/TokenVector.Inference/demo/tkv_simd_native.dll', shell=True)
print('copy rc =', r.returncode)
r = subprocess.run('cp /content/TokenVector/Release/tools/simd_native/tkv_simd_native.dll /content/TokenVector.Inference/tkv_simd_native.dll', shell=True)
print('copy2 rc =', r.returncode)
r = subprocess.run('cd /content/TokenVector/Release/tools/simd_native && gcc -O2 -mavx2 -mfma -shared -fPIC tkv_step124.c -o tkv_step124.dll', shell=True, capture_output=True, text=True)
print('step124 rc =', r.returncode)
if r.returncode != 0: print(r.stderr[-2000:])
r = subprocess.run('cp /content/TokenVector/Release/tools/simd_native/tkv_step124.dll /content/TokenVector.Inference/demo/tkv_step124.dll', shell=True)
print('copy3 rc =', r.returncode)
import ctypes
try:
    lib = ctypes.CDLL('/content/TokenVector.Inference/demo/tkv_simd_native.dll')
    lib.TkvCUDA_Available.restype = ctypes.c_int
    print('TkvCUDA_Available() =', lib.TkvCUDA_Available())
except Exception as e:
    print('load native failed:', e)


In [ ]:
# 4/5 Build tkv_torch_fused.so (ATen bridge)
import subprocess, os, sys
torch_dir = subprocess.check_output([sys.executable, '-c', 'import torch,os; print(os.path.dirname(torch.__file__))']).decode().strip()
print('torch:', torch_dir)
incs = [torch_dir + '/include', torch_dir + '/include/torch/csrc/api/include']
libdir = torch_dir + '/lib'
cmd = ('cd /content/TokenVector/Release/tools/simd_native && '
       'g++ -O2 -shared -std=c++20 -fPIC torch_fused.cpp -o tkv_torch_fused.so '
       '-I' + incs[0] + ' -I' + incs[1] + ' -L' + libdir + ' '
       '-Wl,-rpath,' + libdir + ' '
       '-ltorch -ltorch_cpu -lc10 -lc10_cuda -ltorch_cuda')
p = subprocess.run(cmd, shell=True, capture_output=True, text=True, timeout=1800)
print('build rc =', p.returncode)
if p.returncode != 0:
    print('STDERR:', p.stderr[-4000:])
print('STDOUT tail:', (p.stdout or '')[-1500:])
so_path = '/content/TokenVector/Release/tools/simd_native/tkv_torch_fused.so'
print('exists:', os.path.exists(so_path))
os.environ['TKV_TORCH_PATH'] = so_path
os.environ['LD_LIBRARY_PATH'] = libdir + ':' + os.environ.get('LD_LIBRARY_PATH', '')
import ctypes
try:
    lib = ctypes.CDLL(so_path)
    print('tkv_torch_fused.so: ctypes load OK')
except Exception as e:
    print('ctypes load fail:', e)


In [ ]:
# 5/5 Check attention backward có chạy CUDA không?
import subprocess
p = subprocess.run('cd TokenVector.Inference && printf "1024\n2\n1\n" > tv/tests/_seq_cfg.txt && TKV_TORCH_GPU=1 /opt/dotnet/dotnet demo/bseq.dll', shell=True, capture_output=True, text=True, timeout=1800)
for line in ((p.stdout or '') + '\n' + (p.stderr or '')).splitlines():
    if any(k in line for k in ['BPHASE', '[tkv-attn-bwd]', 'FAIL', 'Exception']):
        print(line.rstrip())


In [ ]:
# 6/5 Bench: .tkv GPU vs torch CUDA (xen ke x3, median)
import subprocess, re, statistics as st
DOT = '/opt/dotnet/dotnet'
def run(cmd):
    p = subprocess.run(cmd, shell=True, capture_output=True, text=True, timeout=1800)
    return (p.stdout or '') + '\n' + (p.stderr or '')
def tkv_gpu():
    out = run('cd TokenVector.Inference && printf "1024\n2\n1\n" > tv/tests/_seq_cfg.txt && TKV_TORCH_GPU=1 ' + DOT + ' demo/bseq.dll')
    m = re.search(r'fwd=([\d.]+) loss=([\d.]+) bwd=([\d.]+) opt=([\d.]+) copyback=([\d.]+) TONG=([\d.]+)', out)
    if not m: raise RuntimeError('BPHASE không tìm thấy: ' + out[-500:])
    return dict(zip(('fwd', 'ce', 'bwd', 'opt', 'copyback', 'total'), map(float, m.groups())))
def torch_cuda():
    out = run('cd TokenVector.Inference && TKV_BENCH_DEVICE=cuda python3 benchmark/bench_torch.py d 1024')
    fw = re.search(r'fwd\+bwd seq1024 dev=cuda: median=([\d.]+)', out)
    fu = re.search(r'full step seq1024 dev=cuda: median=([\d.]+)', out)
    return dict(fwdbwd=float(fw.group(1)), full=float(fu.group(1)))
print('Run...')
tk = []; tt = []
for r in range(3):
    if r % 2 == 0: a, b = tkv_gpu(), torch_cuda()
    else: b, a = torch_cuda(), tkv_gpu()
    tk.append(a); tt.append(b)
    print('%d: .tkv fwd=%.0f CE=%.0f bwd=%.0f opt=%.0f TONG=%.0f | torch fwd+bwd=%.1f full=%.1f' % (r+1, a['fwd'], a['ce'], a['bwd'], a['opt'], a['total'], b['fwdbwd'], b['full']))
med = st.median
fb, tb = med([x['fwd']+x['ce']+x['bwd'] for x in tk]), med([x['total'] for x in tk])
tfb, ttf = med([x['fwdbwd'] for x in tt]), med([x['full'] for x in tt])
print()
print('--- MEDIAN ---')
print('.tkv fwd+bwd   : %.1f ms | torch fwd+bwd   : %.1f ms | %.2fx' % (fb, tfb, fb/tfb))
print('.tkv full step : %.1f ms | torch full step : %.1f ms | %.2fx' % (tb, ttf, tb/ttf))


In [ ]:
# 7/5 Tách OP (TKV_PROF=1)
DOT = '/opt/dotnet/dotnet'
def run(cmd):
    p = subprocess.run(cmd, shell=True, capture_output=True, text=True, timeout=1800)
    for line in ((p.stdout or '') + '\n' + (p.stderr or '')).splitlines():
        if any(k in line for k in ['BPHASE', 'TKV_PROF', 'LN_', 'ATTN_', 'LIN_', 'EMB_', 'BIAS']):
            print(line.rstrip())
print('=== native ===')
run('cd TokenVector.Inference && printf "1024\n2\n1\n" > tv/tests/_seq_cfg.txt && TKV_TORCH_PATH= TKV_PROF=1 ' + DOT + ' demo/bseq.dll')
print('=== GPU bridge ===')
run('cd TokenVector.Inference && printf "1024\n2\n1\n" > tv/tests/_seq_cfg.txt && TKV_TORCH_GPU=1 TKV_PROF=1 ' + DOT + ' demo/bseq.dll')


In [ ]:
# 8/5 Peak RAM native
import subprocess
p = subprocess.run('cd TokenVector.Inference && printf "1024\n0\n1\n" > tv/tests/_seq_cfg.txt && TKV_TORCH_PATH= /usr/bin/time -v /opt/dotnet/dotnet demo/bseq.dll 2>&1 | grep -E "BSEQ|Maximum resident"', shell=True, capture_output=True, text=True, timeout=1800)
print(p.stdout)
